In [1]:
# pip install pyscf scipy
import numpy as np
from functools import reduce
from pyscf import gto, scf, ao2mo, fci
from pyscf.fci import direct_spin1
from scipy.sparse.linalg import LinearOperator, eigsh  # or use your own propagator

In [3]:
# --- 1) Molecule + RHF
mol = gto.M(
    atom="H 0 0 0; H 0 0 0.74",  # change geometry here
    basis="sto-3g",
    charge=0,
    spin=0,
)
mf = scf.RHF(mol).run()

# --- 2) One-/two-electron integrals in MO basis
C = mf.mo_coeff
h1 = reduce(np.dot, (C.T, mf.get_hcore(), C))  # (norb, norb)
eri_ao = mol.intor("int2e")  # (nao, nao, nao, nao) in 8-fold symmetry
eri_mo = ao2mo.incore.full(eri_ao, C)  # packed
norb = C.shape[1]
eri = ao2mo.restore(1, eri_mo, norb)  # (norb, norb, norb, norb)
ecore = mol.energy_nuc()  # add as scalar shift if you want total E

# --- 3) FCI space dimensions (alpha/beta strings)
nalpha, nbeta = mol.nelec
na = fci.cistring.num_strings(norb, nalpha)
nb = fci.cistring.num_strings(norb, nbeta)
dim = na * nb
ci_shape = (na, nb)


# --- 4) Hamiltonian matvec on CI vector (flattened)
def H_dot(vec):
    ci = vec.reshape(ci_shape)
    hci = direct_spin1.contract_1e(h1, ci, norb, (nalpha, nbeta))
    hci += direct_spin1.contract_2e(eri, ci, norb, (nalpha, nbeta))
    # if you want TOTAL energy operator include nuclear repulsion:
    # hci += ecore * ci
    return hci.ravel()

converged SCF energy = -1.11675930739643


In [5]:
H = LinearOperator((dim, dim), matvec=H_dot, dtype=float)

# --- 5) Initial state (HF determinant in FCI space)
# ci0 = fci.addons.init_guess_1e(h1, norb, (nalpha, nbeta))
ci0 = np.zeros((na, nb))
ci0[0, 0] = 1.0
psi0 = ci0.ravel() / np.linalg.norm(ci0)

# Example A) Get ground state via Lanczos/Davidson (you can swap for your propagator)
e0, v0 = eigsh(H, k=1, which="SA", v0=psi0, maxiter=200)
E_ground_elec = float(e0[0])
E_ground_total = E_ground_elec + ecore
psi_ground = v0[:, 0]  # normalized CI vector (flattened)

print("Electronic E0 =", E_ground_elec, "  Total E0 =", E_ground_total)
# psi_ground is your wavefunction state vector in the determinant basis.

# Example B) One imaginary-time step with your MVP (first-order, for illustration):
tau = 0.01
psi = psi0.copy()
psi = psi - tau * H_dot(psi)  # |ψ> ← (I - τH)|ψ>
psi /= np.linalg.norm(psi)

Electronic E0 = 0.4786472774475147   Total E0 = 1.193751616528596


/var/folders/x9/8zgqvcbx66q2mjvw0wjdx_fw0000gn/T/ipykernel_21203/1167524671.py:28: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  hci += direct_spin1.contract_2e(eri, ci, norb, (nalpha, nbeta))
/var/folders/x9/8zgqvcbx66q2mjvw0wjdx_fw0000gn/T/ipykernel_21203/3862479910.py:21: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  psi = psi - tau * H_dot(psi)          # |ψ> ← (I - τH)|ψ>


In [6]:
psi

array([ 9.99993359e-01,  5.04967396e-19,  5.04967396e-19, -3.64440525e-03])

In [7]:
H

<4x4 _CustomLinearOperator with dtype=float64>

In [3]:
# pip install pyscf openfermion openfermionpyscf
import numpy as np
from functools import reduce
from pyscf import gto, scf, ao2mo
from openfermion import (
    FermionOperator,
    QubitOperator,
    jordan_wigner,
    hermitian_conjugated,
)
from openfermion.transforms import normal_ordered

# pip install rdkit pyscf
from rdkit import Chem
from pyscf import gto

In [5]:
# --- 1) Load the molecule from SDF
mol_rd = Chem.MolFromMolFile(
    "../biocomplex/C2H6S2.sdf", removeHs=False
)  # keep explicit Hs
conf = mol_rd.GetConformer()

# --- 2) Build atom string for PySCF
atoms = []
for atom in mol_rd.GetAtoms():
    pos = conf.GetAtomPosition(atom.GetIdx())
    sym = atom.GetSymbol()
    atoms.append(f"{sym} {pos.x:.6f} {pos.y:.6f} {pos.z:.6f}")

# atoms is a list like ['C 0.6424 0.5887 -0.3704', 'C -0.5710 0.3606 0.5213', ...]
atom_str = "; ".join(atoms)

# --- 3) Pass to PySCF
mol = gto.M(atom=atom_str, basis="sto-3g", charge=0, spin=0)
mf = mol.RHF().run()

converged SCF energy = -864.606187993551


In [6]:
# ----- 1) Molecule + RHF
# mol = gto.M(atom='H 0 0 0; H 0 0 0.74', basis='sto-3g', charge=0, spin=0)
# mf = scf.RHF(mol).run()
C = mf.mo_coeff
h1 = reduce(np.dot, (C.T, mf.get_hcore(), C))  # (norb, norb)
eri_ao = mol.intor("int2e")
eri_mo = ao2mo.incore.full(eri_ao, C)
norb = C.shape[1]
eri = ao2mo.restore(1, eri_mo, norb)  # (p,q,r,s)
ecore = mol.energy_nuc()

# ----- 2) Build second-quantized fermionic Hamiltonian
# H = \sum_{pq} h1[p,q] a_p^\dag a_q + 0.5 * \sum_{pqrs} g[p,q,r,s] a_p^\dag a_q^\dag a_r a_s + ecore
Hf = FermionOperator()
for p in range(norb):
    for q in range(norb):
        if abs(h1[p, q]) > 1e-12:
            Hf += FermionOperator(((p, 1), (q, 0)), h1[p, q])
for p in range(norb):
    for q in range(norb):
        for r in range(norb):
            for s in range(norb):
                g = 0.5 * eri[p, q, r, s]
                if abs(g) > 1e-12:
                    Hf += FermionOperator(((p, 1), (q, 1), (r, 0), (s, 0)), g)
# Make Hermitian & normal ordered; add nuclear repulsion
Hf = normal_ordered(Hf + hermitian_conjugated(Hf)) * 0.5
Hf += FermionOperator((), ecore)

# ----- 3) Map to qubits (Jordan–Wigner)
Hq: QubitOperator = jordan_wigner(Hf)
Hq.compress(1e-12)

# ----- 4) Extract Pauli terms
# Hq is a sum: sum_j c_j * P_j ; each P_j is a tuple like ( (q,'X'), (k,'Z'), ... )
terms = [(pauli_term, coeff) for pauli_term, coeff in Hq.terms.items()]


# ----- 5) Commutation test for Pauli strings
def pauli_commute(P, Q):
    # P, Q are tuples like ((q,'X'), (k,'Z'), ...)
    # Build dicts for quick axis lookup
    dP = dict(P)
    dQ = dict(Q)
    # Count sites with non-identity on BOTH and different axes
    cnt = 0
    for i in set(dP) & set(dQ):
        if dP[i] != dQ[i]:
            cnt += 1
    # Commute iff even number of such mismatches
    return (cnt % 2) == 0


# ----- 6) Greedy coloring into commuting groups
groups = []
for P, c in terms:
    placed = False
    for g in groups:
        if all(pauli_commute(P, Pg) for Pg, _ in g):
            g.append((P, c))
            placed = True
            break
    if not placed:
        groups.append([(P, c)])

print(f"#qubits = {Hq.n_qubits if hasattr(Hq, 'n_qubits') else 2 * norb}")
print(f"#Pauli terms = {len(terms)}")
print(f"#commuting groups (greedy) = {len(groups)}")

# Each group is mutually commuting; you can Trotterize group-wise:
# e^{-iΔt H} ≈ ∏_groups exp(-iΔt ∑_{(P,c)∈group} c P)
# then within a group factor further or diagonalize blockwise if desired.

#qubits = 68
#Pauli terms = 1157
#commuting groups (greedy) = 64
